<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L06-agent-client-protocol/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L06-agent-client-protocol/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L06-agent-client-protocol/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L06-agent-client-protocol/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# Build an Agent Client Protocol agent's session lifecycle and drive it from three toy editors

**You will build:** five functions that let one scripted coding agent speak the Agent Client
Protocol (ACP) over JSON-RPC: the `initialize` exchange, a permission round trip, a response
matcher for a wire where both sides send requests, a prompt turn that streams and can be
cancelled, and a test that tells ACP's messages from MCP's.
**Time:** ~50 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, offline · **Prerequisites:** T07-L02-mcp-from-the-wire-up

By the end you will be able to:
1. Implement the `initialize` exchange: protocol-version negotiation and a capability record in which every omitted capability counts as unsupported.
2. Implement the permission round trip, deciding allow, reject or cancelled from an option's kind rather than its id, and failing closed on a malformed reply.
3. Implement a response matcher for a connection on which both sides send requests, so a cancel notification or a request from the editor that arrives while you wait is handled and never mistaken for the reply.
4. Implement a prompt turn that streams updates, gates file access on the editor's advertised capabilities, asks permission before an edit, and ends with the stop reason `cancelled`, never an error, when the editor cancels.
5. Run the same unchanged agent against three toy editors and classify the messages on a mixed wire as ACP or MCP by what they say.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L02`

**This lesson assumes you can already:**

- tell a JSON-RPC request, notification and response apart by their fields (id, method, result, error), as T07-L02 does for MCP
- read a small Python class with state and call its methods (the lesson hands you Conn, Session and AgentState to use, not to write), and loop over and index into lists of dicts
- raise and catch your own exception class with try and except
- read an assert-based check and use its failure message to find the bug in a function

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [JSON-RPC 2.0 Specification](https://www.jsonrpc.org/specification) · JSON-RPC Working Group · *free* — the short base this lesson sits on: requests and notifications, ids, and the error codes the agent returns
- [Agent Client Protocol: Overview](https://agentclientprotocol.com/protocol/v1/overview) · Agent Client Protocol project · *free* — the message flow, the agent and client methods and the conventions the three editors and the classifier in exercise 5 are built from
- [Agent Client Protocol: Prompt Turn](https://agentclientprotocol.com/protocol/v1/prompt-turn) · Agent Client Protocol project · *free* — the session updates, stop reasons and cancellation rules that exercise 4 implements, with the JSON examples
- [The Python Tutorial, section 8: Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html) · Python Software Foundation · *free* — try, except and defining your own exception class, which is how RpcError travels between the wire and the code that waits for a reply
- [The Python Tutorial, section 9: Classes](https://docs.python.org/3/tutorial/classes.html) · Python Software Foundation · *free* — objects, methods and instance state, which is how the given Conn, Session and AgentState hold what the agent remembers and what your run_turn calls

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib, copy, io, json, os, signal, sys, threading, time, traceback
from collections import Counter, deque
from typing import Any, Callable, Optional

print("python", sys.version.split()[0], "· standard library only")


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful: whoever runs the notebook may
# already have an alarm running (tools/grade.py stops every rubric row after 45 s and repeats
# its alarm every 0.25 s; a harness may have an alarm that ends the process). Earlier guards
# switched that alarm off on the way out (`setitimer(ITIMER_REAL, 0)`), so the NEXT hang ran for
# ever. This one saves the running alarm, lets whichever deadline comes first win (handing the
# signal to the outer handler when it is the outer's turn), and on the way out re-arms the outer
# alarm with the time it had left and its interval. tools/verify_all.py fails a lesson whose
# guard does less.
# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 10


class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the function each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("handle_initialize",),
    "exercise 2": ("interpret_permission",),
    "exercise 3": ("await_response",),
    "exercise 4": ("run_turn",),
    "exercise 5": ("classify_wire_message",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (await_response)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped: needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped: needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet: fill in {stub}() above, then re-run this cell.")
        else:
            print(f"{label}: skipped: {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED: {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Two protocols, two sides of the agent

An AI coding agent has two conversations to run. One is with the **tools and data** it uses: a
database, a search index, a file service. That is the Model Context Protocol (MCP), which you
built the wire of in T07-L02. The other is with the **editor** the user is sitting in: the
editor sends the user's prompt, and the agent streams its progress back, asks before it changes a
file, and can be told to stop. That is the **Agent Client Protocol (ACP)**.

```text
  editor (ACP client)  <--- ACP: JSON-RPC over the agent's stdin and stdout --->  agent
                                                                                    |
                                         MCP: JSON-RPC to each tool server  <-------+
```

ACP re-uses JSON-RPC 2.0 (a specification last updated in 2013, which does not move week to week; ACP does) and, over its stdio transport, the framing you already know: one UTF-8
JSON message per line, the editor launching the agent as a subprocess. What is new is the
vocabulary and one structural fact: **both sides send requests**. The editor asks the agent to
start a prompt turn; in the middle of that turn the agent asks the editor for permission, or for
the unsaved text of a buffer. And the editor can interrupt with a one-way notification. ACP also lets either
side add methods of its own, named with a leading underscore; an agent that has never heard of one answers
with the standard "Method not found" error and carries on.

**Which revision.** ACP is young and still moving. As of 2026-10-24 the protocol's documentation
presents version 1 as the latest (a version 2 is a draft), and the newest published schema
release is `schema-v1.26.0` (`claims.yaml`); the wire shapes below were checked against `schema-v1.24.1` and the
two later releases change none of them (they add optional session notices and compaction, which this lesson does not use,
and make a few receiver checks stricter). Every `protocolVersion` in this lesson is the
integer 1; the revision numbers can change in a later release without that integer changing,
because the integer names only a MAJOR, breaking version.

In [ ]:
# Given: this lesson's constants, error type and framing. Run it; nothing here is graded.
SPEC_AS_OF = "2026-10-24"          # the day the specification pages were read (claims.yaml)
PROTOCOL_VERSION = 1               # the one major version of ACP this agent speaks
SUPPORTED_VERSIONS = (PROTOCOL_VERSION,)
AGENT_INFO = {"name": "scripted-agent", "title": "Scripted agent", "version": "0.1.0"}
STOP_REASONS = ("end_turn", "max_tokens", "max_turn_requests", "refusal", "cancelled")
TOOL_STATUSES = ("pending", "in_progress", "completed", "failed")


class RpcError(Exception):
    """A JSON-RPC error: a code and a message, raised and caught on both sides of the wire."""

    def __init__(self, code: int, message: str):
        super().__init__(f"{code}: {message}")
        self.code, self.message = code, message


def encode(msg: dict) -> bytes:
    """One message, one line: compact UTF-8 JSON and a newline. json.dumps escapes any newline
    inside a string, so the framing rule (no embedded newlines) holds by construction."""
    return json.dumps(msg, separators=(",", ":"), ensure_ascii=False).encode("utf-8") + b"\n"


def _valid_id(value: Any) -> bool:
    return isinstance(value, (str, int)) and not isinstance(value, bool)


def kind_of(msg: Any) -> str:
    """'request', 'notification', 'response' or 'invalid'. A notification is a request without an id."""
    if not isinstance(msg, dict) or msg.get("jsonrpc") != "2.0":
        return "invalid"
    if "method" in msg:
        if not isinstance(msg["method"], str):
            return "invalid"
        if "id" not in msg:
            return "notification"
        return "request" if _valid_id(msg["id"]) else "invalid"
    if "id" in msg and ("result" in msg) != ("error" in msg):
        return "response"
    return "invalid"


def decode(line: bytes) -> dict:
    """One line to one message. Raises RpcError(-32700) for text that is not JSON and
    RpcError(-32600) for JSON that is not a JSON-RPC message."""
    try:
        msg = json.loads(line.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError):
        raise RpcError(-32700, "Parse error") from None
    if kind_of(msg) == "invalid":
        raise RpcError(-32600, "Invalid Request")
    return msg


print("protocol version", PROTOCOL_VERSION, "· specification read on", SPEC_AS_OF)
_line = encode({"jsonrpc": "2.0", "method": "session/cancel", "params": {"sessionId": "sess_x"}})
print("one notification on the wire:", _line)
print("the same line decoded, and what kind of message it is:", kind_of(decode(_line)))

## 2. The agent, the session and the wire

The next cell is the machinery around your five functions. Read it once; you will not edit it.

- `AgentState` and `Session` hold what the agent remembers: whether `initialize` has happened,
  what the editor said it can do, and one record per conversation. `session.cancelled` is set
  the moment a `session/cancel` notification arrives.
- `LoopWire` is the pair of pipes, in memory. The agent reads lines with `readline()` and writes
  them with `write()`, exactly the two operations a real child process performs on its stdin and
  stdout. An in-memory wire gives the same bytes on every machine and every run, and lets the
  lesson count them. A real editor would put a child process and real pipes here: the lines would be the same,
  but this lesson never ran that leg, so flushing, partial reads and a child that exits are all untested here.
- `Conn` is the agent's end of the connection: it frames messages, numbers the agent's own
  requests, and routes whatever arrives. While your code waits for a reply it hands everything
  else to `Conn.on_other`, which records a cancel, answers a request from the editor with an
  error, and never lets a stray message derail the wait. Its two error codes of its own, -31001 (`session/new` before
  `initialize`) and -31003 (`initialize`, `session/new` or a second `session/prompt` while a turn runs), are this lesson's choice from
  the space JSON-RPC leaves to applications, outside its reserved block -32768 to -32000. ACP's schema says it uses -32000 to -32099
  for protocol-specific errors (it already gives -32000 to "authentication required" and -32002 to "resource not found", and -32800 to
  "request cancelled"), so a code of the lesson's own there could clash with a later release (`claims.yaml`).
- `handle_prompt` is the glue for `session/prompt`: it marks the session busy, calls **your**
  `run_turn`, and turns what you return into the response.

Ids are the one subtle point. A JSON-RPC id is chosen by whoever sends the request, so the editor
has its own ids (0, 1, 2 ...) and the agent has its own (also 0, 1, 2 ...). The two sets overlap.

In [ ]:
# Given: the agent's state, the wire and the connection.
class Session:
    """One conversation: its id, working directory, what the agent has read, and whether a
    cancel has arrived."""

    def __init__(self, session_id: str, cwd: str, mcp_servers: list):
        self.id, self.cwd, self.mcp_servers = session_id, cwd, mcp_servers
        self.files: dict[str, str] = {}   # path -> text the agent has read through the editor
        self.cancelled = False            # set by Conn when a session/cancel notification arrives
        self.busy = False                 # True while a prompt turn is running
        self._calls = 0

    def new_call_id(self) -> str:
        self._calls += 1
        return f"call_{self._calls:03d}"


class AgentState:
    """Everything the agent remembers about one connection."""

    def __init__(self):
        self.initialized = False
        self.client_caps: dict = {}       # what the editor advertised in `initialize`
        self.sessions: dict[str, Session] = {}


def client_can(caps: Any, *path: str) -> bool:
    """True only if the editor advertised this capability as the JSON value true. Anything omitted,
    null, false or malformed is unsupported (claims.yaml: omitted capabilities are UNSUPPORTED)."""
    node = caps
    for key in path:
        if not isinstance(node, dict):
            return False
        node = node.get(key)
    return node is True


class LoopWire:
    """The agent's stdin and stdout, in memory, with an editor on the other end. Every line in
    either direction is kept in `log`. `LIMIT` stops a loop that never ends with a named error."""
    LIMIT = 400

    def __init__(self, editor):
        self.editor = editor
        self.inbox: deque = deque()
        self.log: list[tuple[str, bytes]] = []
        self.calls = 0
        for line in editor.start():
            self._arrive(line)

    def _tick(self) -> None:
        self.calls += 1
        if self.calls > self.LIMIT:
            raise AssertionError(f"more than {self.LIMIT} reads and writes on one connection: something is "
                                 "looping. Check that every wait ends when a reply or the end of the stream arrives")

    def _arrive(self, line: bytes) -> None:
        self.log.append(("editor->agent", line))
        self.inbox.append(line)

    def pending(self) -> bool:
        return bool(self.inbox)

    def readline(self) -> bytes:
        """The next line the editor sent, or b'' once the editor has nothing more (stdin closed)."""
        self._tick()
        return self.inbox.popleft() if self.inbox else b""

    def write(self, line: bytes) -> None:
        """What the agent writes to stdout. The editor reacts at once; its answers queue up for the agent."""
        self._tick()
        assert line.endswith(b"\n") and line.count(b"\n") == 1, "stdout must carry exactly one JSON message per line"
        self.log.append(("agent->editor", line))
        for out in self.editor.receive(line):
            self._arrive(out)

    def decoded(self) -> list[tuple[str, dict]]:
        """The whole conversation as (direction, message). A line that is not a message is kept raw."""
        rows = []
        for direction, line in self.log:
            try:
                rows.append((direction, json.loads(line)))
            except ValueError:
                rows.append((direction, {"_raw": line}))
        return rows


class Conn:
    """The agent's end of the connection."""

    def __init__(self, wire, state: AgentState):
        self.wire, self.state = wire, state
        self.next_id = 0                      # the AGENT's own request ids: a separate space from the editor's
        self.stray: list[dict] = []           # responses nobody was waiting for
        self.internal_errors: list[str] = []

    # --- sending -----------------------------------------------------------------------
    def write(self, msg: dict) -> None:
        self.wire.write(encode(msg))

    def notify(self, method: str, params: dict) -> None:
        self.write({"jsonrpc": "2.0", "method": method, "params": params})

    def request(self, method: str, params: dict) -> dict:
        """Send a request to the editor and wait for its reply. Returns the result; raises RpcError
        if the editor answers with an error, ConnectionError if the stream ends first."""
        request_id = self.next_id
        self.next_id += 1
        self.write({"jsonrpc": "2.0", "id": request_id, "method": method, "params": params})
        return await_response(self.read_message, request_id, self.on_other)

    def reply(self, request_id: Any, result: dict) -> None:
        self.write({"jsonrpc": "2.0", "id": request_id, "result": result})

    def reply_error(self, request_id: Any, code: int, message: str) -> None:
        self.write({"jsonrpc": "2.0", "id": request_id, "error": {"code": code, "message": message}})

    def client_can(self, *path: str) -> bool:
        return client_can(self.state.client_caps, *path)

    # --- receiving ---------------------------------------------------------------------
    def read_message(self) -> Optional[dict]:
        """Block for the next valid message. A line that is not JSON-RPC is answered with the
        JSON-RPC error and skipped. Returns None when the editor has closed the stream."""
        while True:
            line = self.wire.readline()
            if not line:
                return None
            try:
                return decode(line)
            except RpcError as exc:
                self.write({"jsonrpc": "2.0", "id": None, "error": {"code": exc.code, "message": exc.message}})

    def poll(self) -> None:
        """Handle every message that has already arrived, without waiting for any more."""
        while self.wire.pending():
            msg = self.read_message()
            if msg is None:
                return
            self.on_other(msg)

    def _cancel(self, msg: dict) -> None:
        params = msg.get("params")
        session_id = params.get("sessionId") if isinstance(params, dict) else None
        session = self.state.sessions.get(session_id) if isinstance(session_id, str) else None
        if session is not None and session.busy:     # a cancel with no turn running is ignored
            session.cancelled = True

    def on_other(self, msg: dict) -> None:
        """Everything that arrives while the agent is busy waiting for something else."""
        kind = kind_of(msg)
        if kind == "notification":
            if msg["method"] == "session/cancel":
                self._cancel(msg)
        elif kind == "request":
            if msg["method"] in ("initialize", "session/new", "session/prompt"):
                self.reply_error(msg["id"], -31003, "this agent runs one prompt turn at a time")
            else:
                self.reply_error(msg["id"], -32601, "Method not found")
        else:
            self.stray.append(msg)

    def handle(self, msg: dict) -> None:
        """Route one message that arrived while the agent was idle."""
        kind = kind_of(msg)
        if kind == "notification":
            if msg["method"] == "session/cancel":
                self._cancel(msg)
            return
        if kind == "response":
            self.stray.append(msg)
            return
        handlers = {"initialize": lambda p: handle_initialize(self.state, p),
                    "session/new": lambda p: handle_session_new(self.state, p),
                    "session/prompt": lambda p: handle_prompt(self, p)}
        handler = handlers.get(msg["method"])
        if handler is None:
            self.reply_error(msg["id"], -32601, "Method not found")
            return
        params = msg.get("params")
        try:
            result = handler(params if isinstance(params, dict) else {})
        except RpcError as exc:
            self.reply_error(msg["id"], exc.code, exc.message)
        except (NotImplementedError, AssertionError, ConnectionError):
            raise
        except Exception as exc:                      # a bug in a handler is an error reply, not a dead agent
            self.internal_errors.append(f"{type(exc).__name__}: {exc}")
            self.reply_error(msg["id"], -32603, f"Internal error: {type(exc).__name__}")
        else:
            self.reply(msg["id"], result)


def serve(conn: Conn) -> None:
    """The agent's main loop: read a message, route it, until the editor closes the stream."""
    while True:
        msg = conn.read_message()
        if msg is None:
            return
        conn.handle(msg)

## 3. Exercise 1: `initialize`

The first thing an editor sends is `initialize`, with two things the agent must act on: the
**protocol version** it wants and the **capabilities** it offers (what it can do for the agent:
read and write text files, run terminals).

The rules from the specification (`claims.yaml`):

- If the agent supports the requested version it answers with that same version. Otherwise it answers
  with the **latest version it supports**, and the editor decides whether it can live with that.
- Every capability the editor omits is **unsupported**. The agent must not call a method whose
  capability was not advertised.
- The agent's answer carries `protocolVersion`, `agentCapabilities`, `agentInfo` and `authMethods`.
  It must advertise only what it implements: this agent has no `session/load`, no images, no audio.

The function also records the outcome in `state`: `state.client_caps` (a copy of the capabilities,
or an empty dict when there are none) and `state.initialized = True`. A request whose
`protocolVersion` is missing or is not an integer raises `RpcError(-32602, ...)` and leaves `state`
untouched. JSON `true` is not an integer.

<details><summary>💡 Hint 1: what to think about</summary>

What should an editor asking for a version below everything you support get? For one above? Is "the
smaller of the two" the same rule as "the latest I support"? And what would an editor that later edits
its own capability dictionary do to your record if you stored it without copying?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Validate first, so a bad request changes nothing. Then: same version if it is in `supported`, else
`max(supported)`. Store a deep copy of the capabilities when they form a JSON object, and an empty
dictionary otherwise. Build the answer from `AGENT_INFO`, copying any dictionary you hand out.
</details>

In [ ]:
def handle_initialize(state: AgentState, params: dict, supported: tuple = SUPPORTED_VERSIONS) -> dict:
    """Answer an `initialize` request and record what the editor can do.

    Example:
        >>> st = AgentState()
        >>> handle_initialize(st, {"protocolVersion": 1, "clientCapabilities": {"fs": {"readTextFile": True}}})["protocolVersion"]
        1
        >>> st.client_caps, st.initialized
        ({'fs': {'readTextFile': True}}, True)

    Returns the result dict: protocolVersion, agentCapabilities (loadSession False, promptCapabilities
    image/audio/embeddedContext all False), agentInfo, authMethods []. Raises RpcError(-32602, ...)
    for a missing or non-integer protocolVersion, leaving `state` unchanged.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_initialize() -> None:
    st = AgentState()
    res = handle_initialize(st, {"protocolVersion": 1, "clientCapabilities": {"fs": {"readTextFile": True}}})
    assert res["protocolVersion"] == 1, "an editor asking for a version the agent supports gets that version back"
    assert st.client_caps == {"fs": {"readTextFile": True}} and st.initialized is True, \
        "record the editor's capabilities in state.client_caps and set state.initialized"
    assert res["agentCapabilities"]["loadSession"] is False, \
        "advertise only what the agent implements: it has no session/load, so loadSession is False"
    assert res["authMethods"] == [] and res["agentInfo"]["name"], "the answer carries authMethods and agentInfo"
    assert handle_initialize(AgentState(), {"protocolVersion": 2})["protocolVersion"] == 1, \
        "an unsupported version is answered with the latest version this agent supports, not echoed back"
    assert handle_initialize(AgentState(), {"protocolVersion": 0}, supported=(1, 2))["protocolVersion"] == 2, \
        "below the supported range is also unsupported: answer with the latest supported version"
    fresh = AgentState()
    handle_initialize(fresh, {"protocolVersion": 1})
    assert fresh.client_caps == {}, "no clientCapabilities means no capabilities: an empty dict"
    for bad in ({}, {"protocolVersion": "1"}, {"protocolVersion": True}, {"protocolVersion": None}):
        st = AgentState()
        try:
            handle_initialize(st, bad)
        except RpcError as exc:
            assert exc.code == -32602 and not st.initialized, f"{bad!r}: invalid params is -32602, and state must stay untouched"
        else:
            raise AssertionError(f"{bad!r} has no integer protocolVersion: raise RpcError(-32602, ...)")
    print("exercise 1 looks right")


_try("exercise 1", _check_initialize)

In [ ]:
# Given: session/new and session/prompt, and the scripted brain of this agent. The agent has exactly
# one trick: a fixed list of steps. What it does at each step is up to `run_turn`, which you write.
def handle_session_new(state: AgentState, params: dict) -> dict:
    if not state.initialized:
        raise RpcError(-31001, "call initialize before session/new")
    cwd = params.get("cwd")
    absolute = isinstance(cwd, str) and (cwd.startswith("/") or (len(cwd) > 2 and cwd[1] == ":" and cwd[2] in "\\/"))
    if not absolute:
        raise RpcError(-32602, "Invalid params: cwd must be an absolute path")
    servers = params.get("mcpServers", [])
    if not isinstance(servers, list):
        raise RpcError(-32602, "Invalid params: mcpServers must be a list")
    session = Session(f"sess_{len(state.sessions) + 1:04d}", cwd, copy.deepcopy(servers))
    state.sessions[session.id] = session
    return {"sessionId": session.id}


NOTES, SETTINGS = "/work/notes.md", "/work/settings.toml"
SCRIPT = [
    {"op": "say", "text": "I will read both files, then make one edit to each."},
    {"op": "read", "path": NOTES},
    {"op": "edit", "path": NOTES, "append": "- tidied by the agent\n"},
    {"op": "read", "path": SETTINGS},
    {"op": "edit", "path": SETTINGS, "append": "# tidied by the agent\n"},
    {"op": "say", "text": "That is everything I was asked to do."},
]
PERMISSION_OPTIONS = [
    {"optionId": "opt-allow", "name": "Allow once", "kind": "allow_once"},
    {"optionId": "opt-reject", "name": "Reject", "kind": "reject_once"},
]


def handle_prompt(conn: Conn, params: dict) -> dict:
    session = conn.state.sessions.get(params.get("sessionId")) if isinstance(params.get("sessionId"), str) else None
    if session is None:
        raise RpcError(-32602, "Invalid params: unknown sessionId")
    blocks = params.get("prompt")
    if not (isinstance(blocks, list) and blocks and all(isinstance(b, dict) and b.get("type") in ("text", "resource_link")
                                                       for b in blocks)):
        raise RpcError(-32602, "Invalid params: prompt must be a non-empty list of text or resource_link blocks")
    session.busy, session.cancelled = True, False
    try:
        reason = run_turn(conn, session, SCRIPT)
    finally:
        session.busy = False
    if reason not in STOP_REASONS:
        raise RpcError(-32603, f"run_turn returned {reason!r}, which is not a stop reason")
    session.cancelled = False
    return {"stopReason": reason}


# Given: the small senders a turn is built from. Each one is a single notification or request.
def say(conn: Conn, session: Session, text: str) -> None:
    conn.notify("session/update", {"sessionId": session.id, "update": {
        "sessionUpdate": "agent_message_chunk", "content": {"type": "text", "text": text}}})


def announce(conn: Conn, session: Session, call_id: str, title: str, kind: str, path: str) -> None:
    """Tell the editor a tool call exists, so it can show it, before anything is asked or done."""
    conn.notify("session/update", {"sessionId": session.id, "update": {
        "sessionUpdate": "tool_call", "toolCallId": call_id, "title": title, "kind": kind,
        "status": "pending", "locations": [{"path": path}]}})


def report(conn: Conn, session: Session, call_id: str, status: str, text: str = "", diff: Optional[tuple] = None) -> None:
    """Change a tool call's status (pending, in_progress, completed, failed), with optional output."""
    assert status in TOOL_STATUSES, f"{status!r} is not a tool-call status"
    update: dict = {"sessionUpdate": "tool_call_update", "toolCallId": call_id, "status": status}
    content = []
    if text:
        content.append({"type": "content", "content": {"type": "text", "text": text}})
    if diff:
        path, old, new = diff
        content.append({"type": "diff", "path": path, "oldText": old, "newText": new})
    if content:
        update["content"] = content
    conn.notify("session/update", {"sessionId": session.id, "update": update})


def fs_read(conn: Conn, session: Session, path: str) -> str:
    """Ask the editor for a file's text, unsaved changes included. Raises RpcError if it refuses."""
    text = conn.request("fs/read_text_file", {"sessionId": session.id, "path": path}).get("content")
    if not isinstance(text, str):
        raise RpcError(-32603, "the editor's reply carried no text")
    return text


def fs_write(conn: Conn, session: Session, path: str, content: str) -> None:
    conn.request("fs/write_text_file", {"sessionId": session.id, "path": path, "content": content})

## 4. Exercise 2: reading the permission reply

Before it changes a file the agent sends the editor a `session/request_permission` request with a
list of **options**. Each option has an `optionId` the agent made up, a human-readable name, and a
**kind**: `allow_once`, `allow_always`, `reject_once` or `reject_always`. The editor answers:

```text
{"outcome": {"outcome": "selected", "optionId": "opt-allow"}}      the user chose an option
{"outcome": {"outcome": "cancelled"}}                             the prompt turn was cancelled
```

`interpret_permission(options, result)` turns the reply into one of three words: `"allow"`,
`"reject"` or `"cancelled"`. Two rules decide it. **What an option means is its kind, never its
id:** the agent chose the ids, and an id such as `"allow"` may sit on a reject option. And **a reply
you cannot understand is a refusal**: an agent that edits a file on a malformed reply has failed
open, and the file is the user's. Match the protocol's words exactly too: `canceled` is not `cancelled`, and `Selected` is not `selected`;
a word you do not recognise is a reply you cannot read.

<details><summary>💡 Hint 1: what to think about</summary>

Where does the word "allow" legitimately live in the reply? Which of the two fields tells you what
the user decided: the id you invented, or the kind you attached to it? What should a reply that is not
an object, an id that no option carries, or an id that is not even a string, produce?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Dig into `result["outcome"]` carefully, checking each level is a dictionary. `cancelled` is its own
answer. For `selected`, build a lookup from each option's id to its kind, look the chosen id up, and
call it allow only for the two allow kinds. Everything else, including every malformed shape, is reject.
</details>

In [ ]:
def interpret_permission(options: list, result: Any) -> str:
    """Decide what the editor's reply to session/request_permission means.

    Example:
        >>> opts = [{"optionId": "a", "name": "Yes", "kind": "allow_once"}, {"optionId": "b", "name": "No", "kind": "reject_once"}]
        >>> interpret_permission(opts, {"outcome": {"outcome": "selected", "optionId": "a"}})
        'allow'

    Returns "allow", "reject" or "cancelled". Never raises, whatever `result` looks like.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_permission() -> None:
    opts = [{"optionId": "yes", "name": "Do it", "kind": "reject_once"},
            {"optionId": "no", "name": "Skip it", "kind": "allow_always"},
            {"optionId": "x", "name": "Never", "kind": "reject_always"}]
    pick = lambda oid: {"outcome": {"outcome": "selected", "optionId": oid}}
    assert interpret_permission(opts, pick("no")) == "allow", \
        "the option with kind allow_always is an allow, whatever its id says; read the kind, not the id"
    assert interpret_permission(opts, pick("yes")) == "reject", "the option with kind reject_once is a reject, even though its id is 'yes'"
    assert interpret_permission(opts, pick("x")) == "reject", "reject_always is a reject"
    assert interpret_permission(opts, {"outcome": {"outcome": "cancelled"}}) == "cancelled", \
        "a cancelled outcome is its own answer, neither allow nor reject"
    for bad in (None, {}, [], {"outcome": "selected"}, {"outcome": {"outcome": "selected"}}, pick("nope"),
                pick(["no"]), pick(None), {"outcome": {"outcome": "shrug"}}):
        assert interpret_permission(opts, bad) == "reject", \
            f"{bad!r} is not a usable answer: an unclear reply must fail closed as 'reject' and must never raise"
    print("exercise 2 looks right")


_try("exercise 2", _check_permission)

## 5. Exercise 3: waiting for a reply on a wire where both sides talk

When the agent sends a request it has to wait for **that request's response**. On a wire where only
one side sends requests, that is the next line. Here it is not: while the agent waits for the editor to
answer a permission question, the editor may send a `session/cancel` notification, or a request of its
own, and both arrive before the answer does.

`await_response(read_message, request_id, on_other)` reads messages until it meets the response to
`request_id` and then returns its result. Everything else goes at once to `on_other(msg)`, which handles it
(it is how a cancel reaches the session) and returns. Four rules:

- A response is a message **without a `method`** whose `id` equals `request_id`. An incoming
  *request* can carry the same id number: it is the editor's id, in the editor's id space, and it is
  not the answer.
- A response with an `error` raises `RpcError(code, message)`.
- `read_message()` returns `None` once the other side has closed the stream: raise `ConnectionError`.
  Waiting for ever is never an option.
- JSON `false` is not the number 0, although Python says `False == 0`.

<details><summary>💡 Hint 1: what to think about</summary>

Which parts of a message tell a response from a request and from a notification? What does a response
to someone else's request (a different id) deserve? What ends the loop other than the answer arriving?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Loop: read a message; no message means the stream ended. Decide whether it is *the* response: it has no
`method`, it has a `result` or an `error`, and its id is the one you sent (and is not a boolean).
If it is not, pass it to `on_other` and keep waiting. If it is, an error becomes `RpcError`, a result is returned.
</details>

In [ ]:
def await_response(read_message: Callable[[], Optional[dict]], request_id: int, on_other: Callable[[dict], None]) -> dict:
    """Wait for the response to the request you sent with `request_id`.

    Example (a cancel notification arrives first, then the answer):
        >>> msgs = iter([{"jsonrpc": "2.0", "method": "session/cancel", "params": {}},
        ...              {"jsonrpc": "2.0", "id": 0, "result": {"ok": True}}])
        >>> seen = []
        >>> await_response(lambda: next(msgs), 0, seen.append)
        {'ok': True}
        >>> [m["method"] for m in seen]
        ['session/cancel']

    Returns the response's result. Raises RpcError for an error response and ConnectionError when
    read_message() returns None before the response arrives.
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _CountedReader:
    """Feeds scripted messages to await_response, then None (stream closed). A wait that ignores
    the end of the stream fails with a message instead of hanging the notebook."""

    def __init__(self, messages: list, limit: int = 50):
        self.messages, self.limit, self.reads = deque(messages), limit, 0

    def __call__(self) -> Optional[dict]:
        self.reads += 1
        if self.reads > self.limit:
            raise AssertionError("await_response is still reading after the stream closed: when read_message() "
                                 "returns None, raise ConnectionError")
        return self.messages.popleft() if self.messages else None


def _check_await() -> None:
    ok = {"jsonrpc": "2.0", "id": 1, "result": {"v": 7}}
    cancel = {"jsonrpc": "2.0", "method": "session/cancel", "params": {"sessionId": "s"}}
    other: list = []
    assert await_response(_CountedReader([cancel, ok]), 1, other.append) == {"v": 7}, "return the result of the matching response"
    assert other == [cancel], "a message that is not the response goes to on_other, in order, and the wait goes on"
    clash = {"jsonrpc": "2.0", "id": 1, "method": "_x/focus", "params": {}}
    other = []
    assert await_response(_CountedReader([clash, ok]), 1, other.append) == {"v": 7} and other == [clash], \
        "a REQUEST with id 1 is the editor's own id, not the answer to your id 1: a response has no 'method'"
    other = []
    stale = {"jsonrpc": "2.0", "id": 9, "result": {}}
    assert await_response(_CountedReader([stale, ok]), 1, other.append) == {"v": 7} and other == [stale], \
        "a response with a different id is not yours: pass it to on_other"
    try:
        await_response(_CountedReader([{"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}]), 1, other.append)
    except RpcError as exc:
        assert exc.code == -32601 and exc.message == "Method not found", "carry the error's code and message into RpcError"
    else:
        raise AssertionError("an error response should raise RpcError")
    try:
        await_response(_CountedReader([cancel]), 1, other.append)
    except ConnectionError:
        pass
    else:
        raise AssertionError("when the stream ends before the reply, raise ConnectionError")
    print("exercise 3 looks right")


_try("exercise 3", _check_await)

In [ ]:
# Given: three toy editors. They are three different clients of the same protocol. Each is a small state
# machine that reacts to every line the agent writes; none of them knows how the agent is built.
START_BUFFERS = {NOTES: "# Notes\n- ship the lesson\n", SETTINGS: 'mode = "draft"\n'}
PROMPT_TEXT = "Tidy the notes and the settings file."
EDITOR_MCP_SERVERS = [{"name": "notes-index", "command": "/usr/local/bin/notes-index-server", "args": ["--stdio"], "env": []}]


class ToyEditor:
    """A minimal ACP client. initialize, then session/new, then one session/prompt; it answers the
    agent's requests, records what it sees, and counts every way the agent breaks the protocol."""
    name = "toy editor"
    client_caps: Optional[dict] = None          # what to advertise in `initialize`; None leaves the field out

    def __init__(self):
        self.buffers = dict(START_BUFFERS)      # the editor's own, possibly unsaved, text
        self.waiting: dict[int, str] = {}       # our request id -> method
        self.next_id = 0
        self.session_id: Optional[str] = None
        self.stop_reason: Optional[str] = None
        self.updates: list[dict] = []
        self.tool_calls: dict[str, dict] = {}   # toolCallId -> what we know of it
        self.violations: list[str] = []
        self.cancel_sent = False
        self.custom_replies: list[dict] = []

    # --- helpers ---------------------------------------------------------------------
    def _request(self, method: str, params: dict) -> bytes:
        rid, self.next_id = self.next_id, self.next_id + 1
        self.waiting[rid] = method
        return encode({"jsonrpc": "2.0", "id": rid, "method": method, "params": params})

    def _result(self, rid: Any, result: dict) -> bytes:
        return encode({"jsonrpc": "2.0", "id": rid, "result": result})

    def _error(self, rid: Any, code: int, message: str) -> bytes:
        return encode({"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": message}})

    def _advertises(self, *path: str) -> bool:
        return client_can(self.client_caps, *path)

    # --- the conversation ------------------------------------------------------------
    def start(self) -> list[bytes]:
        params: dict = {"protocolVersion": PROTOCOL_VERSION, "clientInfo": {"name": self.name, "version": "0.0.1"}}
        if self.client_caps is not None:
            params["clientCapabilities"] = self.client_caps
        return [self._request("initialize", params)]

    def receive(self, line: bytes) -> list[bytes]:
        msg = decode(line)
        kind = kind_of(msg)
        if kind == "response":
            return self._on_response(self.waiting.pop(msg["id"], None), msg)
        if kind == "notification":
            return self._on_update(msg["params"]) if msg["method"] == "session/update" else []
        return self._on_request(msg)

    def _on_response(self, method: Optional[str], msg: dict) -> list[bytes]:
        if method is None:
            self.custom_replies.append(msg)                   # an answer to a request we never sent as such
            return []
        if "error" in msg:
            if method.startswith("_"):
                self.custom_replies.append(msg)
            else:
                self.violations.append(f"{method} was answered with an error: {msg['error']['message']}")
            return []
        if method == "initialize":
            if msg["result"].get("protocolVersion") != PROTOCOL_VERSION:
                self.violations.append("the agent chose a protocol version this editor does not speak")
            return [self._request("session/new", {"cwd": "/work", "mcpServers": EDITOR_MCP_SERVERS})]
        if method == "session/new":
            self.session_id = msg["result"]["sessionId"]
            return [self._request("session/prompt", {"sessionId": self.session_id,
                                                      "prompt": [{"type": "text", "text": PROMPT_TEXT}]})]
        if method == "session/prompt":
            self.stop_reason = msg["result"].get("stopReason")
        return []

    def _on_update(self, params: dict) -> list[bytes]:
        update = params["update"]
        self.updates.append(update)
        if params.get("sessionId") != self.session_id:
            self.violations.append("a session/update named a session this editor never opened")
        if update["sessionUpdate"] == "tool_call":
            self.tool_calls[update["toolCallId"]] = dict(update)
        elif update["sessionUpdate"] == "tool_call_update":
            if update["toolCallId"] not in self.tool_calls:
                self.violations.append("tool_call_update for a tool call that was never announced")
            else:
                self.tool_calls[update["toolCallId"]].update(update)
        return []

    def _on_request(self, msg: dict) -> list[bytes]:
        method, rid, params = msg["method"], msg["id"], msg.get("params") or {}
        if self.cancel_sent:
            self.violations.append(f"the agent sent {method} after the turn was cancelled")
        if method in ("fs/read_text_file", "fs/write_text_file"):
            cap = "readTextFile" if method == "fs/read_text_file" else "writeTextFile"
            if not self._advertises("fs", cap):
                self.violations.append(f"the agent called {method} without the {cap} capability")
                return [self._error(rid, -32601, "Method not found")]
            path = params.get("path")
            if method == "fs/read_text_file":
                if path not in self.buffers:
                    return [self._error(rid, -32002, "Resource not found")]
                return [self._result(rid, {"content": self.buffers[path]})]
            self.buffers[path] = params["content"]
            return [self._result(rid, {})]
        if method == "session/request_permission":
            call = self.tool_calls.get(params.get("toolCall", {}).get("toolCallId"))
            if call is None:
                self.violations.append("session/request_permission named a tool call that was never announced")
                return [self._result(rid, {"outcome": {"outcome": "cancelled"}})]
            return self.decide(rid, params, call)
        return [self._error(rid, -32601, "Method not found")]

    def decide(self, rid: Any, params: dict, call: dict) -> list[bytes]:
        raise NotImplementedError

    def choose(self, rid: Any, params: dict, kind_wanted: str) -> bytes:
        """Answer with the first option of the wanted kind. Editors pick by kind, as the protocol intends."""
        option = next(o for o in params["options"] if o["kind"] == kind_wanted)
        return self._result(rid, {"outcome": {"outcome": "selected", "optionId": option["optionId"]}})


class BufferEditor(ToyEditor):
    """Editor A: reads and writes through its own buffers. The user allows edits to Markdown notes
    and refuses edits to settings files. Before it answers its first question it sends a request of its
    own, numbered with the very id the agent is waiting on."""
    name = "A: buffer editor"
    client_caps = {"fs": {"readTextFile": True, "writeTextFile": True}}

    def decide(self, rid, params, call):
        out = []
        if not getattr(self, "_nudged", False):
            self._nudged = True
            out.append(encode({"jsonrpc": "2.0", "id": rid, "method": "_acme/focus_file",
                               "params": {"path": call["locations"][0]["path"]}}))
        wanted = "allow_once" if call["locations"][0]["path"].endswith(".md") else "reject_once"
        return out + [self.choose(rid, params, wanted)]


class ViewerEditor(ToyEditor):
    """Editor B: a read-only viewer. It advertises no capabilities at all (the field is left out)."""
    name = "B: read-only viewer"
    client_caps = None

    def decide(self, rid, params, call):
        return [self.choose(rid, params, "allow_once")]


class ImpatientEditor(ToyEditor):
    """Editor C: full file access, but the user presses Stop the moment the first question appears.
    It sends the session/cancel notification, then answers the question with the cancelled outcome."""
    name = "C: impatient editor"
    client_caps = {"fs": {"readTextFile": True, "writeTextFile": True}}

    def decide(self, rid, params, call):
        self.cancel_sent = True
        return [encode({"jsonrpc": "2.0", "method": "session/cancel", "params": {"sessionId": self.session_id}}),
                self._result(rid, {"outcome": {"outcome": "cancelled"}})]


EDITORS = (BufferEditor, ViewerEditor, ImpatientEditor)


def run_session(editor: ToyEditor) -> tuple:
    """Connect `editor` to a fresh agent, run the whole conversation, return (state, wire, conn)."""
    state = AgentState()
    wire = LoopWire(editor)
    conn = Conn(wire, state)
    serve(conn)
    return state, wire, conn

## 6. Exercise 4: the prompt turn

The agent has a script (`SCRIPT`, above): say something, read a file, edit it, read another, edit
it, say something. `run_turn(conn, session, script)` plays that script for one `session/prompt` and
returns the **stop reason**: `"end_turn"` when it finishes, `"cancelled"` when the editor cancelled.
The senders `say`, `announce`, `report`, `fs_read` and `fs_write` are given. You write the control flow:

- **Before every step** call `conn.poll()`; if `session.cancelled`, return `"cancelled"` at once.
  `conn.client_can("fs", "readTextFile")` is the given test for "the editor advertised this". The specification says a capability that is "false or not present"
  is unsupported; this helper is stricter, a convention of this lesson: only the JSON value `true` counts, so `"yes"` or `1` is unsupported (the published schema agrees: it types both as booleans that default to false when a value cannot be read). That suits the two
  boolean `fs` capabilities used here; some ACP capabilities, such as `elicitation`, are objects with rules of their own.
- `say`: send the text. `read`: announce a `read` tool call, then read through the editor, remember the
  text in `session.files[path]` and report `completed`. `edit`: announce an `edit` tool call, then ask permission,
  then (only if allowed) report `in_progress`, write the old text plus the step's `append`, remember it, report `completed` with a diff.
- **Capabilities.** No `fs.readTextFile` advertised: the read tool call is reported `failed`, no `fs/*` request is sent. Likewise
  a missing `fs.writeTextFile` for an edit, and then **no permission is asked**: do not ask the user to approve what you cannot do.
  An edit of a file you have not read is `failed` too.
- **Permission.** `conn.request("session/request_permission", {"sessionId", "toolCall": {"toolCallId", "title",
  "kind", "locations"}, "options": PERMISSION_OPTIONS})`. Reject: report `failed` and carry on with the next step.
- **Cancellation, the rule that matters.** If the reply is `cancelled`, **or** a cancel arrived while you waited
  (`session.cancelled`, even when the reply says allow), write nothing and return `"cancelled"`; a `cancelled` reply counts on its own, even
  when it comes before the `session/cancel` notification or with none at all. If the editor
  answers a request with an error and `session.cancelled` is set, that is the cancellation showing through: return `"cancelled"`,
  never an error. With no cancel, an `RpcError` just fails that one tool call.

<details><summary>💡 Hint 1: what to think about</summary>

In what order must the editor learn things: that a tool call exists, that permission is wanted for it, that it
is running, that it finished? At which points can a cancel have arrived without you noticing yet? What must
be true of the file before and after a write you were told to abort?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

One loop over the script. Top of the loop: poll, then check `session.cancelled`. A helper per step kind keeps
the loop short. Around each request: `try`/`except RpcError`, and inside the handler ask
`session.cancelled` before deciding between `failed` and `cancelled`. After the permission reply, look at
both the verdict and the flag before writing. Finish with `"end_turn"`.
</details>

In [ ]:
def run_turn(conn: Conn, session: Session, script: list) -> str:
    """Play `script` for one prompt turn and return the stop reason.

    Example (an editor that allows everything and never cancels):
        >>> run_turn(conn, session, [{"op": "say", "text": "hello"}])
        'end_turn'

    Returns "end_turn" or "cancelled". Sends no request to the editor after a cancel has arrived.
    Never lets an RpcError escape: a failed request fails its tool call (or means "cancelled").
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _run(editor_cls) -> tuple:
    editor = editor_cls()
    state, wire, conn = run_session(editor)
    return editor, state, wire, conn


def _agent_requests(wire: LoopWire) -> list[str]:
    return [m["method"] for d, m in wire.decoded() if d == "agent->editor" and kind_of(m) == "request"]


def _check_turn() -> None:
    editor, state, wire, conn = _run(BufferEditor)
    assert editor.stop_reason == "end_turn", \
        f"a turn that runs its whole script ends with 'end_turn', not {editor.stop_reason!r} (internal errors: {conn.internal_errors})"
    assert editor.buffers[NOTES] == START_BUFFERS[NOTES] + "- tidied by the agent\n", "the allowed edit should reach the editor's buffer, appended to what the agent read"
    assert editor.buffers[SETTINGS] == START_BUFFERS[SETTINGS], "the user refused the settings edit: nothing may be written there"
    assert _agent_requests(wire) == ["fs/read_text_file", "session/request_permission", "fs/write_text_file",
                                     "fs/read_text_file", "session/request_permission"], \
        "per file: read, then ask, then (if allowed) write; the refused edit ends with no write"
    assert not editor.violations, f"the editor saw the agent break the protocol: {editor.violations}"
    editor, state, wire, conn = _run(ViewerEditor)
    assert _agent_requests(wire) == [], "an editor that advertised no capabilities gets no fs/* request and no permission question"
    assert editor.stop_reason == "end_turn" and not editor.violations, f"the turn still ends cleanly: {editor.stop_reason!r} {editor.violations}"
    editor, state, wire, conn = _run(ImpatientEditor)
    assert editor.stop_reason == "cancelled", f"after session/cancel the prompt must be answered with stopReason 'cancelled', not {editor.stop_reason!r}"
    assert editor.buffers == START_BUFFERS and not editor.violations, \
        f"nothing may be written after a cancel and no request may follow it: {editor.violations}"

    class _RepliesFirst(BufferEditor):
        """Answers the first question 'cancelled' and only then sends session/cancel."""
        def decide(self, rid, params, call):
            self.cancel_sent = True
            return [self._result(rid, {"outcome": {"outcome": "cancelled"}}),
                    encode({"jsonrpc": "2.0", "method": "session/cancel", "params": {"sessionId": self.session_id}})]

    editor, state, wire, conn = _run(_RepliesFirst)
    assert editor.stop_reason == "cancelled" and editor.buffers == START_BUFFERS and not editor.violations, \
        ("a 'cancelled' permission reply stops the turn by itself, even before the session/cancel notification arrives: "
         f"write nothing and send nothing more (stop reason {editor.stop_reason!r}, {editor.violations})")

    class _StopsAtOnce(BufferEditor):
        """Presses Stop right after the agent's first sentence, while the agent is not waiting for anything."""
        def _on_update(self, params):
            super()._on_update(params)
            if self.cancel_sent or params["update"]["sessionUpdate"] != "agent_message_chunk":
                return []
            self.cancel_sent = True
            return [encode({"jsonrpc": "2.0", "method": "session/cancel", "params": {"sessionId": self.session_id}})]

    editor, state, wire, conn = _run(_StopsAtOnce)
    assert editor.stop_reason == "cancelled" and _agent_requests(wire) == [], \
        "a cancel can arrive while you are not waiting for anything: look for it before every step, and send no request after it"
    print("exercise 4 looks right")


_try("exercise 4", _check_turn, needs=("exercise 1", "exercise 2", "exercise 3"))

### The same agent, three editors

Now the point of a protocol. The cell below runs one agent, unchanged, against each editor, and counts what
happened from the wire itself. First editor A's whole conversation, one line per message. Editor A sends one
request of its own while the agent is waiting for an answer: find it, and see how the agent replied.

In [ ]:
def brief(msg: dict) -> str:
    """One line for a message: its kind, id and method, and for an update its sessionUpdate and status."""
    kind = kind_of(msg)
    if kind == "request":
        return f"request  #{msg['id']} {msg['method']}"
    if kind == "notification":
        upd = (msg.get("params") or {}).get("update") or {}
        tail = f"  [{upd['sessionUpdate']}" + (f", {upd['status']}" if "status" in upd else "") + "]" if "sessionUpdate" in upd else ""
        return f"notify       {msg['method']}{tail}"
    if kind == "response":
        return f"response #{msg['id']} " + (f"error {msg['error']['code']}" if "error" in msg else "result")
    return "(not a message)"


def _show_transcript() -> None:
    editor, state, wire, conn = _run(BufferEditor)
    for direction, msg in wire.decoded():
        print(f"{'editor -> agent' if direction.startswith('editor') else 'agent -> editor'}  {brief(msg)}")
    ids_agent = sorted(m["id"] for d, m in wire.decoded() if d == "agent->editor" and kind_of(m) == "request")
    ids_editor = sorted(m["id"] for d, m in wire.decoded() if d == "editor->agent" and kind_of(m) == "request")
    clash = sorted(set(ids_agent) & set(ids_editor))
    print(f"\nrequest ids the agent used: {ids_agent}; the editor used: {ids_editor}; both sides used: {clash}")
    print(f"answers to a request the agent never made, set aside by the agent: {len(conn.stray)}")


_try("transcript of editor A", _show_transcript, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

In [ ]:
def _matrix() -> None:
    header = f"{'editor':<22}{'stop reason':<13}{'updates':>8}{'asked':>7}{'fs calls':>9}  {'tool calls ok/failed':<22}{'buffers changed':<17}{'violations'}"
    print(header)
    rows = []
    for cls in EDITORS:
        editor, state, wire, conn = _run(cls)
        to_editor = [m for d, m in wire.decoded() if d == "agent->editor"]
        updates = sum(1 for m in to_editor if m.get("method") == "session/update")
        asked = sum(1 for m in to_editor if m.get("method") == "session/request_permission")
        fs_calls = sum(1 for m in to_editor if str(m.get("method")).startswith("fs/"))
        status = Counter(c.get("status") for c in editor.tool_calls.values())
        changed = [p.rsplit("/", 1)[1] for p in editor.buffers if editor.buffers[p] != START_BUFFERS[p]]
        rows.append((editor, state, conn))
        print(f"{editor.name:<22}{str(editor.stop_reason):<13}{updates:>8}{asked:>7}{fs_calls:>9}  "
              f"{status['completed']} ok / {status['failed']} failed{'':<7}{', '.join(changed) or '-':<17}{len(editor.violations)}")
    print()
    for editor, state, conn in rows:
        servers = sum(len(s.mcp_servers) for s in state.sessions.values())
        print(f"{editor.name:<22}MCP servers the editor handed to the agent: {servers}; agent internal errors: {len(conn.internal_errors)}")


_try("three editors", _matrix, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

Read the table as a user would. Which editor had an edit applied, and which was refused one? Which was never
asked a question, and did its turn still end normally? Which pressed Stop, and did the agent report an error or
a stop reason? Not one line of the agent changed between rows. That is what an interop protocol is for. The three editors are toy clients
written for this lesson: none is a real editor, and nothing here tests how any real editor behaves.

The MCP-servers count in the second block is the bridge to MCP. The editor handed the agent the address of a
tool server in `session/new`. The editor never talks to that server; **the agent does**, in MCP, on a wire of its own.

## 7. Exercise 5: the two-column test

Look at a stream of JSON-RPC messages and say which protocol each one belongs to. Most are decided by
the method name:

| column | method names (as of 2026-10-24, `claims.yaml`) |
|---|---|
| ACP: editor and agent | `session/...`, `fs/...`, `terminal/...`, `authenticate` |
| MCP: agent and tool server | `tools/...`, `resources/...`, `prompts/...`, `notifications/...` |

Two names do not settle it. **`initialize`** exists in both, and what tells them apart is the
`protocolVersion` in `params`: ACP's is an **integer** (`1`); MCP's, in the handshake revisions
(2025-11-25 and earlier), is a **date string**. And **`elicitation/create`** is defined by both protocols, so
the name alone cannot say which one the message belongs to.

`classify_wire_message(msg)` returns `"acp"`, `"mcp"`, `"both"` or `"neither"`. A response has no method, so
it is `"neither"`; so is any method in neither column, and an `initialize` whose version is neither an
integer nor a string. A name belongs to a family only with its slash: `sessions/list` is not `session/...`.

The table is this lesson's reduced list, not either specification in full. ACP also has `logout` and `elicitation/complete`, and MCP has names
of its own such as `ping`; all of them are `"neither"` here, because this test knows only the families in the table. MCP's current revision
(2026-07-28, as of 2026-10-24) has no `initialize` at all, so the `initialize` rule above recognises the legacy handshake only.

<details><summary>💡 Hint 1: what to think about</summary>

What does Python think `isinstance(True, int)` is? Which exact characters make a prefix a prefix?
What should a non-dictionary, or a message with no method, be?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Get the method (anything unusable is `"neither"`). Handle `initialize` and `elicitation/create` by exact
name first, then test the prefixes with their slashes.
</details>

In [ ]:
def classify_wire_message(msg: Any) -> str:
    """Which protocol does this message belong to: 'acp', 'mcp', 'both' or 'neither'?

    Example:
        >>> classify_wire_message({"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {}})
        'mcp'
        >>> classify_wire_message({"jsonrpc": "2.0", "id": 0, "method": "initialize", "params": {"protocolVersion": 1}})
        'acp'

    Returns one of the four words. Never raises, whatever `msg` is.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_classify() -> None:
    def m(method, **params):
        return {"jsonrpc": "2.0", "id": 1, "method": method, "params": params}
    assert classify_wire_message(m("session/prompt")) == "acp" and classify_wire_message(m("fs/read_text_file")) == "acp", \
        "session/..., fs/... (and terminal/...) belong to ACP"
    assert classify_wire_message(m("tools/list")) == "mcp" and classify_wire_message(m("notifications/initialized")) == "mcp", \
        "tools/..., resources/..., prompts/... and notifications/... belong to MCP"
    assert classify_wire_message(m("initialize", protocolVersion=1)) == "acp", "initialize with an integer protocolVersion is ACP"
    assert classify_wire_message(m("initialize", protocolVersion="2025-11-25")) == "mcp", "initialize with a date-string protocolVersion is MCP"
    assert classify_wire_message(m("initialize", protocolVersion=True)) == "neither", "JSON true is not an integer version"
    assert classify_wire_message(m("elicitation/create")) == "both", "both protocols define elicitation/create: say so, do not guess"
    assert classify_wire_message({"jsonrpc": "2.0", "id": 1, "result": {}}) == "neither", "a response has no method to classify"
    assert classify_wire_message(m("sessions/list")) == "neither", "a family needs its slash: sessions/ is not session/"
    assert classify_wire_message("not a message") == "neither" and classify_wire_message(None) == "neither", "never raise on a non-message"
    print("exercise 5 looks right")


_try("exercise 5", _check_classify)

In [ ]:
# Given: the MCP side of a tool-using agent, as the messages on an MCP wire (the legacy
# handshake revision 2025-11-25, two calls, and a question the server puts to the agent). These are samples, written by hand from the specification's examples.
MCP_SAMPLES = [
    {"jsonrpc": "2.0", "id": 0, "method": "initialize", "params": {"protocolVersion": "2025-11-25", "capabilities": {},
                                                                    "clientInfo": {"name": "scripted-agent", "version": "0.1.0"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "search_notes", "arguments": {"query": "lesson"}}},
    {"jsonrpc": "2.0", "id": 0, "method": "elicitation/create", "params": {
        "mode": "form", "message": "Which notebook should I search?",
        "requestedSchema": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}}},
]


def column_report(messages: list) -> dict:
    columns: dict[str, list] = {"acp": [], "mcp": [], "both": [], "neither": []}
    for msg in messages:
        columns[classify_wire_message(msg)].append(msg)
    return columns


def _two_columns() -> None:
    editor, state, wire, conn = _run(BufferEditor)
    acp_wire = [m for d, m in wire.decoded() if "method" in m]        # every request and notification on editor A's wire
    mixed = acp_wire + MCP_SAMPLES
    cols = column_report(mixed)
    print(f"{len(mixed)} messages with a method, {len(acp_wire)} from the editor's wire and {len(MCP_SAMPLES)} from the agent's MCP wire")
    for name, label in (("acp", "ACP column"), ("mcp", "MCP column"), ("both", "in both"), ("neither", "neither")):
        methods = Counter(m["method"] for m in cols[name])
        print(f"  {label:<11} {len(cols[name]):>3}  " + ", ".join(f"{k} x{v}" for k, v in sorted(methods.items())))
    on_editor_wire = column_report(acp_wire)
    print(f"MCP messages that crossed the editor's wire: {len(on_editor_wire['mcp'])}; "
          f"ACP messages that crossed the tool server's wire: {len(column_report(MCP_SAMPLES)['acp'])}")
    initialize_both = [classify_wire_message(m) for m in mixed if m["method"] == "initialize"]
    print("the two initialize messages are told apart by their protocolVersion:", initialize_both)


_try("two columns", _two_columns, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 8. What this lesson left out

- **A real subprocess.** The wire here is in memory. A real editor starts the agent as a child process and
  writes to its stdin; the lines are the same bytes. Only `LoopWire` changes.
- **`session/load`, authentication, terminals, plans, modes, slash commands** and the other optional parts
  of the protocol. This agent advertises none of them (`loadSession` is false).
- **A model.** The brain is a fixed script. Everything graded here is the protocol around a brain, which is the same
  whatever sits behind it.
- **Remote agents.** The specification asks agents and clients to support stdio whenever possible; the HTTP transport was still a draft proposal
  on 2026-10-24 (`claims.yaml`).

## 9. Common mistakes

- **Matching a reply by id alone.** The editor's ids and the agent's ids are separate sets; a *request* may carry the number you are waiting for.
- **Treating a cancellation as a failure.** The editor asked for it, so the prompt's answer is the result `stopReason: "cancelled"`, not a JSON-RPC error.
- **Reading the option's id instead of its kind.** The agent made the ids up; the kind is the protocol's vocabulary.
- **Calling a method the editor never advertised.** Capabilities the editor omitted are unsupported; a polite agent does not even ask.
- **Failing open.** A reply you cannot parse is not consent.
- **Waiting for ever.** Every wait ends when the stream does.

## 10. Self-check

1. An editor's `initialize` carries no `clientCapabilities` field at all. What should the agent do?
   - (a) Assume the editor can do everything, since nothing was ruled out. (b) Treat every capability as unsupported and do without the editor's file methods.
   - (c) Refuse the connection with an error. (d) Ask the editor to send `initialize` again with the field.
2. The agent is waiting for the answer to its own request with id 2. A line arrives with id 2 and a `method`. What is it?
   - (a) The answer, because the ids match. (b) A notification, because it has a method. (c) A malformed message, to be dropped silently.
   - (d) A request from the editor, whose id belongs to the editor's own numbering and has to be answered by the agent.
3. The user presses Stop while the agent is waiting for a permission reply. How does the turn end correctly?
   - (a) With a JSON-RPC error on the prompt request, so the editor shows a failure. (b) With no answer to the prompt request at all.
   - (c) With a normal result on the prompt request whose stop reason is `cancelled`, after any last updates. (d) With a new permission request, to confirm.
4. Which of these belongs in the MCP column and not the ACP column?
   - (a) The agent asking a search server to run one of its tools. (b) The agent asking the editor for the unsaved text of a buffer.
   - (c) The editor telling the agent to stop. (d) The agent asking the user to approve a file edit.
5. `session/cancel` has no id. What does the agent send back for it?
   - (a) A result with an empty object. (b) A method-not-found error. (c) Nothing: a notification is never answered, with a result or with an error.
   - (d) A `session/update` that confirms it.

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

You wrote the protocol half of a coding agent: the part that is the same whichever model sits behind it and
whichever editor sits in front. T07-L04 (agent-to-agent) is the third wire an agent can have; this one is
the wire to a person.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now, so the board is current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_initialize),
                              ("exercise 2", _check_permission),
                              ("exercise 3", _check_await),
                              ("exercise 4", _check_turn),
                              ("exercise 5", _check_classify)):
            _try(_name, _check, needs=("exercise 1", "exercise 2", "exercise 3") if _name == "exercise 4" else ())
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.